# NB02 · Depuración y partición *leakage-safe*

**Entorno recomendado:** CPU, sin acelerador  
**Tiempo estimado:** 15–30 minutos  
**Semilla:** 2026

Este notebook:

1. elimina copias exactas conservando un solo representante;
2. calcula una huella perceptual `dHash` para identificar imágenes casi idénticas;
3. crea grupos visuales dentro de cada clase;
4. construye una nueva partición aproximada 70/15/15 por grupos, con al menos dos imágenes por clase en validación y prueba;
5. normaliza las imágenes a RGB y un máximo de 1024 píxeles;
6. prepara la estructura requerida por `YOLO11-cls`.

> Las particiones originales no se utilizan para entrenar porque NB01 confirmó duplicados que cruzan entrenamiento y prueba.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm.auto import tqdm

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
RAW_DIR = PROJECT_ROOT / '01_DATOS_ORIGINALES'
EXTRACT_DIR = RAW_DIR / 'zenodo_11114885_extracted'
RESULTS_DIR = PROJECT_ROOT / '06_RESULTADOS'
PROCESSED_ROOT = PROJECT_ROOT / '02_DATOS_PROCESADOS' / 'classification_5class_v1'
SPLITS_DIR = PROJECT_ROOT / '03_SPLITS'

INVENTORY_CSV = RESULTS_DIR / 'inventario_imagenes.csv'
if not INVENTORY_CSV.exists():
    raise FileNotFoundError('No se encontró inventario_imagenes.csv. Ejecute NB01.')
for path in [RESULTS_DIR, PROCESSED_ROOT, SPLITS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CLASSES = ['damaged', 'empty', 'overripe', 'ripe', 'unripe']
SPLITS = ['train', 'val', 'test']
TARGET_FRACTIONS = {'train': 0.70, 'val': 0.15, 'test': 0.15}

inventory = pd.read_csv(INVENTORY_CSV)
inventory = inventory[(inventory['valid'] == True) & inventory['class_inferred'].isin(CLASSES)].copy()
print('Imágenes válidas recibidas:', len(inventory))
display(inventory.groupby(['split_inferred', 'class_inferred']).size().rename('n').reset_index())


In [ ]:
# Conservamos un único representante de cada hash exacto.
inventory = inventory.sort_values(['class_inferred', 'sha256', 'relative_path']).reset_index(drop=True)
exact_group_sizes = inventory.groupby('sha256').size()
unique_df = inventory.drop_duplicates('sha256', keep='first').copy().reset_index(drop=True)
unique_df['exact_group_size'] = unique_df['sha256'].map(exact_group_sizes).astype(int)

print('Archivos originales:', len(inventory))
print('Imágenes únicas por SHA-256:', len(unique_df))
print('Copias exactas retiradas:', len(inventory) - len(unique_df))
display(unique_df.groupby('class_inferred').size().rename('n_unique').reset_index())


In [ ]:
def safe_rgb(path):
    with Image.open(path) as im:
        try:
            out = ImageOps.exif_transpose(im).convert('RGB')
        except Exception:
            out = im.convert('RGB')
        return out.copy()


def dhash64(path):
    image = safe_rgb(path).convert('L').resize((9, 8), Image.Resampling.LANCZOS)
    arr = np.asarray(image, dtype=np.int16)
    bits = (arr[:, 1:] > arr[:, :-1]).reshape(-1)
    value = 0
    for bit in bits:
        value = (value << 1) | int(bit)
    return int(value)


hashes = []
for rel in tqdm(unique_df['relative_path'], desc='Calculando dHash'):
    hashes.append(dhash64(EXTRACT_DIR / rel))
unique_df['dhash_int'] = hashes
unique_df['dhash_hex'] = [f'{value:016x}' for value in hashes]

print('Huellas perceptuales calculadas:', len(unique_df))


In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


MAX_DHASH_DISTANCE = 4
uf = UnionFind(len(unique_df))
near_pairs = []

for label in CLASSES:
    indices = unique_df.index[unique_df['class_inferred'] == label].tolist()
    for pos, i in enumerate(indices):
        hi = int(unique_df.at[i, 'dhash_int'])
        for j in indices[pos + 1:]:
            hj = int(unique_df.at[j, 'dhash_int'])
            distance = (hi ^ hj).bit_count()
            if distance <= MAX_DHASH_DISTANCE:
                uf.union(i, j)
                near_pairs.append({
                    'class': label,
                    'index_a': i,
                    'index_b': j,
                    'distance': distance,
                    'path_a': unique_df.at[i, 'relative_path'],
                    'path_b': unique_df.at[j, 'relative_path'],
                })

root_to_group = {}
group_ids = []
for i in range(len(unique_df)):
    root = uf.find(i)
    if root not in root_to_group:
        root_to_group[root] = f'PG{len(root_to_group):04d}'
    group_ids.append(root_to_group[root])
unique_df['perceptual_group'] = group_ids

near_df = pd.DataFrame(near_pairs)
near_df.to_csv(RESULTS_DIR / 'pares_visualmente_cercanos_dhash.csv', index=False)
group_sizes = unique_df.groupby('perceptual_group').size()

print('Pares con distancia dHash ≤', MAX_DHASH_DISTANCE, ':', len(near_df))
print('Grupos perceptuales:', unique_df['perceptual_group'].nunique())
print('Grupos con más de una imagen:', int((group_sizes > 1).sum()))


In [ ]:
def assign_groups_for_class(frame, label, seed):
    rng = random.Random(seed)
    groups = []
    for group_id, group in frame.groupby('perceptual_group'):
        groups.append({'group_id': group_id, 'indices': group.index.tolist(), 'size': len(group)})
    rng.shuffle(groups)
    groups.sort(key=lambda item: item['size'], reverse=True)

    n = len(frame)
    n_val = max(2, int(np.ceil(n * TARGET_FRACTIONS['val'])))
    n_test = max(2, int(np.ceil(n * TARGET_FRACTIONS['test'])))
    n_train = n - n_val - n_test
    if n_train < 1:
        raise RuntimeError(f'La clase {label} no tiene imágenes suficientes para tres subconjuntos.')
    targets = {'train': n_train, 'val': n_val, 'test': n_test}
    counts = {split: 0 for split in SPLITS}
    assignment = {}

    # Primero garantizamos presencia de la clase en los tres subconjuntos.
    for split, group in zip(SPLITS, groups[:3]):
        assignment[group['group_id']] = split
        counts[split] += group['size']

    for group in groups[3:]:
        deficits = {split: targets[split] - counts[split] for split in SPLITS}
        split = max(SPLITS, key=lambda s: (deficits[s], -counts[s]))
        assignment[group['group_id']] = split
        counts[split] += group['size']

    return assignment, counts


all_assignments = {}
class_split_counts = []
for class_index, label in enumerate(CLASSES):
    subset = unique_df[unique_df['class_inferred'] == label]
    n_groups = subset['perceptual_group'].nunique()
    if n_groups < 3:
        raise RuntimeError(f'La clase {label} tiene menos de tres grupos independientes.')
    assignment, counts = assign_groups_for_class(subset, label, SEED + class_index)
    all_assignments.update(assignment)
    for split, count in counts.items():
        class_split_counts.append({'class': label, 'split': split, 'n': count})

unique_df['split'] = unique_df['perceptual_group'].map(all_assignments)
if unique_df['split'].isna().any():
    raise RuntimeError('Existen imágenes sin partición asignada.')

split_table = unique_df.groupby(['split', 'class_inferred']).size().rename('n').reset_index()
display(split_table)


In [ ]:
# Comprobaciones de fuga antes de copiar archivos.
sha_leakage = unique_df.groupby('sha256')['split'].nunique()
perceptual_leakage = unique_df.groupby('perceptual_group')['split'].nunique()
assert int((sha_leakage > 1).sum()) == 0, 'Hay hashes exactos en más de un split.'
assert int((perceptual_leakage > 1).sum()) == 0, 'Hay grupos perceptuales en más de un split.'

manifest_columns = [
    'relative_path', 'filename', 'class_inferred', 'split', 'sha256',
    'exact_group_size', 'dhash_hex', 'perceptual_group', 'split_inferred',
    'width', 'height', 'mode'
]
manifest = unique_df[manifest_columns].copy()
manifest.to_csv(SPLITS_DIR / 'palma_split_leakage_safe_seed2026.csv', index=False)
split_table.to_csv(SPLITS_DIR / 'palma_split_distribution_seed2026.csv', index=False)

print('Prueba de fuga exacta: 0 grupos cruzados')
print('Prueba de fuga perceptual: 0 grupos cruzados')
print('Manifiesto guardado:', SPLITS_DIR / 'palma_split_leakage_safe_seed2026.csv')


In [ ]:
def safe_filename(text):
    stem = Path(text).stem
    stem = re.sub(r'[^A-Za-z0-9_-]+', '_', stem).strip('_')
    return stem[:80] or 'image'


processed_rows = []
for row in tqdm(manifest.itertuples(index=False), total=len(manifest), desc='Preparando imágenes'):
    source = EXTRACT_DIR / row.relative_path
    class_dir = PROCESSED_ROOT / row.split / row.class_inferred
    class_dir.mkdir(parents=True, exist_ok=True)
    output_name = f'{row.sha256[:12]}_{safe_filename(row.filename)}.jpg'
    destination = class_dir / output_name

    if not destination.exists():
        image = safe_rgb(source)
        image.thumbnail((1024, 1024), Image.Resampling.LANCZOS)
        image.save(destination, format='JPEG', quality=92, optimize=True)

    processed_rows.append({
        'source_relative_path': row.relative_path,
        'processed_relative_path': str(destination.relative_to(PROJECT_ROOT)),
        'split': row.split,
        'class': row.class_inferred,
        'sha256_source': row.sha256,
        'perceptual_group': row.perceptual_group,
        'processed_bytes': destination.stat().st_size,
    })

processed_manifest = pd.DataFrame(processed_rows)
processed_manifest.to_csv(SPLITS_DIR / 'palma_processed_manifest_seed2026.csv', index=False)
(PROCESSED_ROOT / '.preparation_complete').write_text(
    json.dumps({'seed': SEED, 'n_images': len(processed_manifest)}, indent=2), encoding='utf-8'
)

print('Imágenes procesadas:', len(processed_manifest))
print('Carpeta YOLO11-cls:', PROCESSED_ROOT)
print('Tamaño procesado:', f"{processed_manifest['processed_bytes'].sum()/1024**2:.1f} MB")


In [ ]:
# Validación física final.
physical_counts = []
for split in SPLITS:
    for label in CLASSES:
        folder = PROCESSED_ROOT / split / label
        n_files = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
        expected = int(((processed_manifest['split'] == split) & (processed_manifest['class'] == label)).sum())
        if n_files != expected:
            raise RuntimeError(f'Conteo incorrecto en {split}/{label}: {n_files} != {expected}')
        physical_counts.append({'split': split, 'class': label, 'n': n_files})

physical_df = pd.DataFrame(physical_counts)
display(physical_df.pivot(index='class', columns='split', values='n').fillna(0).astype(int))

pivot = physical_df.pivot(index='class', columns='split', values='n').fillna(0)
ax = pivot[['train', 'val', 'test']].plot(
    kind='bar', figsize=(10, 5), color=['#2e7d32', '#f9a825', '#1565c0']
)
ax.set_title('Partición leakage-safe por clase')
ax.set_xlabel('Clase')
ax.set_ylabel('Número de imágenes únicas')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_leakage_safe_distribucion.png', dpi=220, bbox_inches='tight')
plt.show()

summary = {
    'seed': SEED,
    'n_original_files': int(len(inventory)),
    'n_unique_exact': int(len(unique_df)),
    'exact_copies_removed': int(len(inventory) - len(unique_df)),
    'n_perceptual_groups': int(unique_df['perceptual_group'].nunique()),
    'near_pairs_dhash_le_4': int(len(near_df)),
    'exact_groups_crossing_splits': int((sha_leakage > 1).sum()),
    'perceptual_groups_crossing_splits': int((perceptual_leakage > 1).sum()),
    'processed_dataset': str(PROCESSED_ROOT),
}
(RESULTS_DIR / 'resumen_split_leakage_safe.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(summary, ensure_ascii=False, indent=2))


## Resultado esperado

Al finalizar debe observar:

- `exact_groups_crossing_splits: 0`;
- `perceptual_groups_crossing_splits: 0`;
- carpetas `train`, `val` y `test`, cada una con las cinco clases;
- el manifiesto reproducible de la partición con semilla 2026.

El siguiente notebook será **NB03 · Entrenamiento base con YOLO11-cls** y utilizará GPU.
